In [13]:
# ==========================================
# 11주차: 데이터 로드 및 회귀분석 대상 설정 (code/11_regression.ipynb)
# ==========================================
!pip install pyreadstat statsmodels

from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm
import statsmodels.formula.api as smf

drive.mount('/content/drive')

# 1. 데이터 로드 (10주차와 동일)
DATA_DIR = '/content/drive/MyDrive/hdac_data'
target_cols = [
    'age', 'sex', 'kstrata', 'psu', 'wt_ntr',
    'genertn', 'HEI', 'incm', 'edu', 'BD1_11', 'pa_aerobic'
]

df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols)
if 'HEI' in df.columns:
    df['HE_khei'] = df['HEI']

# 2. 10주차 전처리 동일 적용
df['genertn_rc'] = df['genertn'].apply(lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan))
df['BD1_11_rc'] = df['BD1_11'].replace(9.0, np.nan) # 무응답 결측 처리

# 10주차 기본 분석 대상 (subpop_t1)
cond_adult = df['age'] >= 20
cond_wt = df['wt_ntr'].notnull()
cond_exp = df['genertn_rc'].notnull()
cond_out = df['HE_khei'].notnull()
df['subpop_t1'] = np.where(cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0)

# 3. 11주차 회귀분석 보정변수 완전 사례(Complete Case) 확인
# 최소 보정집합: age, sex, incm, edu
covars = ['age', 'sex', 'incm', 'edu']
cond_covars = df[covars].notnull().all(axis=1)

# 회귀분석 최종 분석 대상 (subpop_reg)
df['subpop_reg'] = np.where((df['subpop_t1'] == 1.0) & cond_covars, 1.0, 0.0)

print('=' * 50)
print(f"1. Table 1 분석 대상자 (subpop_t1 == 1): {int(df['subpop_t1'].sum()):,}명")
print(f"2. 회귀분석 완전사례 대상자 (subpop_reg == 1): {int(df['subpop_reg'].sum()):,}명")
print(f"3. 보정변수 결측으로 인한 제외 인원: {int(df['subpop_t1'].sum() - df['subpop_reg'].sum()):,}명")
print('=' * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Table 1 분석 대상자 (subpop_t1 == 1): 5,816명
2. 회귀분석 완전사례 대상자 (subpop_reg == 1): 5,652명
3. 보정변수 결측으로 인한 제외 인원: 164명


In [14]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

def fit_design_wls_domain(formula, data, domain_col='subpop_reg',
                          weight_col='wt_ntr', strata_col='kstrata', psu_col='psu'):
    dom_mask = (data[domain_col] == 1.0).to_numpy()
    sub = data.loc[dom_mask].copy()

    mod = smf.wls(formula, data=sub, weights=sub[weight_col])
    res = mod.fit()

    X_sub = np.asarray(mod.exog, dtype=float)
    w_sub = np.asarray(mod.weights, dtype=float)
    u_sub = np.asarray(res.resid * w_sub, dtype=float)[:, None] * X_sub

    n_total = len(data)
    p = X_sub.shape[1]
    u_full = np.zeros((n_total, p), dtype=float)
    u_full[dom_mask] = u_sub

    xtwx = (X_sub.T * w_sub) @ X_sub
    bread = np.linalg.pinv(xtwx)

    strata = data[strata_col].to_numpy()
    psu = data[psu_col].to_numpy()

    meat = np.zeros((p, p), dtype=float)
    total_psu = 0
    for h in np.unique(strata):
        idx_h = np.where(strata == h)[0]
        psu_h = psu[idx_h]
        u_h = u_full[idx_h]
        unique_psu_h = np.unique(psu_h)
        m_h = len(unique_psu_h)
        total_psu += m_h
        if m_h > 1:
            psu_sums = np.vstack([u_h[psu_h == c].sum(axis=0) for c in unique_psu_h])
            diff = psu_sums - psu_sums.mean(axis=0, keepdims=True)
            meat += (m_h / (m_h - 1.0)) * (diff.T @ diff)

    cov_design = bread @ meat @ bread
    se = np.sqrt(np.diag(cov_design))
    df_design = total_psu - len(np.unique(strata))

    beta = res.params
    t_stats = beta.to_numpy() / se
    pvals = 2 * stats.t.sf(np.abs(t_stats), df=df_design)
    t_crit = stats.t.ppf(0.975, df=df_design)
    ci_low = beta.to_numpy() - t_crit * se
    ci_high = beta.to_numpy() + t_crit * se

    return pd.DataFrame({
        'beta': beta, 'se': se, 'ci_low': ci_low,
        'ci_high': ci_high, 'p_value': pvals, 'df': df_design
    }, index=beta.index), int(dom_mask.sum())

tab_m1, n_dom1 = fit_design_wls_domain("HE_khei ~ genertn_rc", df)
tab_m2, n_dom2 = fit_design_wls_domain("HE_khei ~ genertn_rc + age + C(sex) + C(incm) + C(edu)", df)

for name, tab, n_obs in [("모형 1 (Unadjusted)", tab_m1, n_dom1),
                          ("모형 2 (Adjusted)", tab_m2, n_dom2)]:
    r = tab.loc['genertn_rc']
    print(f"\n================ {name} 결과 ================")
    print(f"• 회귀계수 (Beta): {r['beta']:.2f}")
    print(f"• 표준오차 (SE): {r['se']:.2f}")
    print(f"• 95% 신뢰구간 (CI): [{r['ci_low']:.2f}, {r['ci_high']:.2f}]")
    print(f"• p-value: {r['p_value']:.4f}")
    print(f"• 분석 인원 (n): {n_obs:,}명")


================ 모형 1 (Unadjusted) 결과 ================
• 회귀계수 (Beta): -2.38
• 표준오차 (SE): 0.81
• 95% 신뢰구간 (CI): [-3.99, -0.78]
• p-value: 0.0039
• 분석 인원 (n): 5,652명

================ 모형 2 (Adjusted) 결과 ================
• 회귀계수 (Beta): -2.49
• 표준오차 (SE): 0.49
• 95% 신뢰구간 (CI): [-3.45, -1.52]
• p-value: 0.0000
• 분석 인원 (n): 5,652명
